# TP — Cas simple avec MCP

L'objectif est d'illustrer le MCP dans un cadre simple. Nous allons nous concentrer uniquement sur deux points :

1. **Decouvrir un MCP**  
   Lister les tools exposés par un serveur MCP, puis appeler un tool directement. Cela permet de comprendre le protocol et de voir ce qui est disponible

2. **MCP + OpenAI**  
   Laisser OpenAI choisir un tool, mais exécuter réellement ce tool via un serveur MCP local.

Le message à retenir :

> MCP sert à standardiser la manière dont des outils sont exposés à une application IA.  
> OpenAI peut choisir un tool, mais c'est notre client MCP qui appelle réellement le serveur MCP.

## 1. Installer et observer un serveur MCP existant

Pour savoir ce qui est disponible en MCP, soit on consulte un registre ou une doc de serveurs MCP, soit on interroge directement le serveur avec list_tools(). Le vrai réflexe MCP, c’est la découverte dynamique des tools.

Commencons par 
https://github.com/cmer81/open-meteo-mcp

### A. Pré-requis

Packages Python :

```bash
pip install -U openai "mcp[cli]"
```

Commandes système utiles :

- `npx` pour lancer des serveurs MCP Node.js ;
- `uvx` pour lancer des serveurs MCP Python sans installation manuelle.

À vérifier dans un terminal :

```bash
node --version
npx --version
uvx --version
```

In [ ]:
!pip install "mcp[cli]"

### B. Fonctions utilitaires MCP

Ces fonctions servent à :

- se connecter à un serveur MCP local via `stdio` ;
- lister les tools exposés ;
- appeler un tool ;
- convertir un résultat MCP en objet Python lisible ;
- convertir un tool MCP local en tool OpenAI function calling.

Des fonctions definissent `async def` parce que BLABLA

In [ ]:

import json
from pprint import pprint

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def get_tool_schema(tool):
    """Récupère le schéma d'entrée d'un tool MCP, selon la version du SDK."""
    return getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None) or {}


def print_tools(tools, show_schema=True):
    for tool in tools:
        print("=" * 90)
        print("Nom :", tool.name)
        print("Description :", tool.description)
        if show_schema:
            print("Schéma d'entrée :")
            pprint(get_tool_schema(tool))


def mcp_result_to_python(result):
    """
    Convertit un résultat MCP en objet Python quand c'est possible.
    Les serveurs MCP peuvent renvoyer du texte, du JSON textuel, ou du contenu structuré.
    """
    structured = getattr(result, "structuredContent", None)
    if structured is not None:
        return structured

    texts = []
    for item in getattr(result, "content", []):
        text = getattr(item, "text", None)
        if text is not None:
            texts.append(text)

    if texts:
        text = "\n".join(texts)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text

    return str(result)


async def list_mcp_tools(server_params: StdioServerParameters):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            return tools_result.tools


async def call_mcp_tool(server_params: StdioServerParameters, tool_name: str, arguments: dict):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments=arguments)
            return mcp_result_to_python(result)


def assistant_message_to_dict(message):
    data = {
        "role": "assistant",
        "content": message.content
    }
    if message.tool_calls:
        data["tool_calls"] = [tool_call.model_dump() for tool_call in message.tool_calls]
    return data


### 2. Serveur météo

Objectif : montrer un serveur MCP local qui donne accès à une vraie API externe, sans authentification.

Le serveur utilisé ici est `open-meteo-mcp-server`, lancé via `npx`. POur l'installer depuis le terminal on fait la chose suivante
```
npx -y -p open-meteo-mcp-server open-meteo-mcp-server
```

Architecture :

```text
Notebook
  ↓ MCP stdio
Open-Meteo MCP server
  ↓
API Open-Meteo
```

Point pédagogique :

> MCP permet de transformer une API météo en tools découvrables et appelables par un client MCP.

In [ ]:
# Pour lancer le serveur météo
open_meteo_params = StdioServerParameters(
    command="npx",
    args=["-y", "-p", "open-meteo-mcp-server", "open-meteo-mcp-server"]
)

In [ ]:
'''
1. Lance le serveur MCP Open-Meteo en sous-processus.
2. Ouvre une session MCP avec lui via stdio.
3. Envoie une requête MCP "list_tools".
4. Récupère la liste des outils exposés par le serveur.
5. Stocke cette liste dans open_meteo_tools.
6. Ferme la session, donc le sous-processus s'arrête.
'''

open_meteo_tools = await list_mcp_tools(open_meteo_params)
print_tools(open_meteo_tools, show_schema=False)

## 1.2 Inspecter quelques schémas d'entrée

Le serveur Open-Meteo expose beaucoup de tools. On cherche ceux qui sont utiles pour :

- géocoder une ville ;
- obtenir une prévision météo.

In [ ]:
def mcp_tool_to_openai_tool(tool):
    """
    Convertit un tool MCP local en tool OpenAI function calling.

    Remarque :
    - Ce n'est pas du MCP remote OpenAI.
    - Ici, OpenAI voit un function tool classique.
    - Le notebook exécute ensuite le tool via MCP.
    """
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": get_tool_schema(tool),
            "strict": False
        }
    }



## 0. Préparation

Packages nécessaires :

```bash
pip install -U openai "mcp[cli]" python-dotenv
```

Dans un environnement de formation, il est préférable de préparer l'environnement avant le TP.

Vous aurez aussi besoin d'une clé OpenAI disponible dans les variables d'environnement :

```bash
OPENAI_API_KEY=sk-...
```

Par exemple dans un fichier `.env`.

In [ ]:
# À exécuter seulement si nécessaire
# %pip install -U openai "mcp[cli]" python-dotenv

## 1. Imports et configuration

On prépare :

- le client OpenAI ;
- les imports MCP ;
- quelques fonctions utilitaires.

In [ ]:
import os
import json
import sys
from pathlib import Path
from pprint import pprint

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openai import OpenAI

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

client = OpenAI()

MODEL = "gpt-4.1-mini"

## 2. Création d'un serveur MCP local

Dans un vrai projet, le serveur MCP pourrait être fourni par une équipe métier, un outil externe, un éditeur logiciel, ou une autre application.

Ici, pour le TP, on crée nous-mêmes un petit serveur MCP local dans un fichier Python.

Ce serveur expose deux tools :

- `calculate_cart_total`
- `divide`

Important :

> Ces tools ne sont pas déclarés directement à OpenAI.  
> Ils sont exposés par un serveur MCP.

In [ ]:
SERVER_FILE = Path("mcp_tp_server.py")

server_code = """
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("tp-mcp-demo")


@mcp.tool()
def calculate_cart_total(
    item_prices: list[float],
    discount_percent: float,
    tax_percent: float,
    shipping_fee: float,
    free_shipping_threshold: float,
) -> dict:
    \"""
    Calculate the final price of a shopping cart.

    Rules:
    - Sum the item prices.
    - Apply the discount percentage.
    - Apply the tax percentage after the discount.
    - Add shipping fee unless the total including tax before shipping is strictly greater than the free shipping threshold.
    - Round values to two decimals.
    \"""
    subtotal = sum(item_prices)
    discounted_total = subtotal * (1 - discount_percent / 100)
    total_with_tax = discounted_total * (1 + tax_percent / 100)

    shipping_applied = 0 if total_with_tax > free_shipping_threshold else shipping_fee
    final_total = total_with_tax + shipping_applied

    return {
        "subtotal": round(subtotal, 2),
        "discounted_total": round(discounted_total, 2),
        "total_with_tax_before_shipping": round(total_with_tax, 2),
        "shipping_applied": round(shipping_applied, 2),
        "final_total": round(final_total, 2),
    }


@mcp.tool()
def divide(a: float, b: float) -> dict:
    \"""
    Divide a by b.

    If b is zero, return a structured error instead of crashing.
    \"""
    if b == 0:
        return {
            "error": "division_by_zero",
            "message": "Division by zero is not defined."
        }

    return {
        "result": a / b
    }


if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

SERVER_FILE.write_text(server_code, encoding="utf-8")

print(f"Serveur MCP créé : {SERVER_FILE.resolve()}")

## 3. Point de vocabulaire

Dans ce TP :

```text
Notebook Jupyter = host
Code Python MCP dans le notebook = client MCP
mcp_tp_server.py = serveur MCP
calculate_cart_total / divide = tools MCP
stdio = transport
```

Le notebook va lancer le serveur MCP en sous-processus, communiquer avec lui, lister ses tools, puis demander l'exécution d'un tool.

# Partie 1 — MCP local sans LLM

Dans cette partie, OpenAI n'intervient pas.

Objectif :

> Comprendre que MCP fonctionne déjà comme un protocole client ↔ serveur, indépendamment du LLM.

## 4. Préparer la connexion au serveur MCP

On indique au client MCP comment lancer le serveur.

Ici, on utilise le transport `stdio`, donc le serveur est lancé comme un processus local.

In [ ]:
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_FILE)]
)

server_params

## 5. Lister les tools exposés par le serveur MCP

On se connecte au serveur, puis on lui demande :

> Quels tools exposes-tu ?

C'est une étape importante : avec MCP, un client peut **découvrir dynamiquement** les capacités d'un serveur.

In [ ]:
def get_tool_schema(tool):
    """Récupère le schema d'entrée d'un tool MCP, avec compatibilité selon versions du SDK."""
    return getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None) or {}


async def list_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            return tools_result.tools


mcp_tools = await list_mcp_tools()

for tool in mcp_tools:
    print("=" * 80)
    print("Nom :", tool.name)
    print("Description :", tool.description)
    print("Schema d'entrée :")
    pprint(get_tool_schema(tool))

### Questions

1. Quels sont les tools exposés par le serveur MCP ?
2. Est-ce que le notebook connaissait ces tools avant d'interroger le serveur ?
3. Quelle information permettrait à un LLM de comprendre quand utiliser chaque tool ?
4. Quelle information permettrait à un programme de valider les arguments ?

## 6. Appeler directement un tool MCP

Maintenant, on appelle un tool sans passer par OpenAI.

On va appeler :

```python
calculate_cart_total(...)
```

mais à travers le protocole MCP.

In [ ]:
def mcp_result_to_python(result):
    """Convertit un résultat MCP en objet Python quand c'est possible."""
    structured = getattr(result, "structuredContent", None)
    if structured is not None:
        return structured

    texts = []
    for item in getattr(result, "content", []):
        text = getattr(item, "text", None)
        if text is not None:
            texts.append(text)

    if texts:
        text = "\n".join(texts)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text

    return str(result)


async def call_mcp_tool(tool_name: str, arguments: dict):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            result = await session.call_tool(tool_name, arguments=arguments)
            return mcp_result_to_python(result)


cart_result = await call_mcp_tool(
    "calculate_cart_total",
    {
        "item_prices": [79.90, 39.50, 249.99],
        "discount_percent": 12,
        "tax_percent": 20,
        "shipping_fee": 8.90,
        "free_shipping_threshold": 350
    }
)

pprint(cart_result)

### Questions

1. Qui a exécuté le calcul ?
2. Est-ce qu'un LLM est intervenu ?
3. Qu'est-ce que MCP a apporté ici ?
4. Quelle différence avec un simple appel Python direct ?

## 7. Appeler un tool qui retourne une erreur structurée

On appelle maintenant `divide` avec une division par zéro.

L'objectif est de montrer qu'un tool peut retourner une erreur métier sans faire planter toute l'application.

In [ ]:
division_result = await call_mcp_tool(
    "divide",
    {
        "a": 10,
        "b": 0
    }
)

pprint(division_result)

### À retenir

Un serveur MCP peut retourner :

- un résultat normal ;
- une erreur structurée ;
- des informations exploitables par le client ou par un LLM.

Cela prépare la logique agentique : le modèle pourra ensuite observer le résultat ou l'erreur, puis décider comment répondre.

# Partie 2 — MCP + OpenAI

Dans cette partie, on ajoute OpenAI.

L'objectif n'est pas de refaire du function calling simple.  
L'objectif est de montrer le pont :

```text
OpenAI choisit un tool
        ↓
Notre client récupère le tool_call
        ↓
Notre client appelle le serveur MCP
        ↓
Le résultat MCP est renvoyé à OpenAI
        ↓
OpenAI formule la réponse finale
```

## 8. Convertir les tools MCP en tools OpenAI

OpenAI attend une liste de tools dans un format spécifique.

MCP expose déjà :

- un nom ;
- une description ;
- un schema d'entrée.

On peut donc construire automatiquement la liste des tools OpenAI à partir des tools MCP.

In [ ]:
def mcp_tool_to_openai_tool(tool):
    schema = get_tool_schema(tool)

    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": schema,
            # Le mode strict est mis à False ici pour rester compatible avec les schemas générés par FastMCP.
            "strict": False
        }
    }


openai_tools = [mcp_tool_to_openai_tool(tool) for tool in mcp_tools]

pprint(openai_tools)

### Question

Dans ce code, est-ce OpenAI qui découvre directement les tools MCP ?

Réponse attendue : non.  
C'est notre notebook, en tant que client MCP, qui interroge le serveur MCP, puis qui traduit les tools MCP dans le format attendu par OpenAI.

## 9. Prompt de test

On reprend un prompt de calcul panier.

Le modèle doit comprendre la demande, choisir le bon tool, extraire les arguments, puis notre client exécutera le tool via MCP.

In [ ]:
SHOP_PROMPT = """
J’ai acheté 3 articles :
- un clavier à 79,90 €
- une souris à 39,50 €
- un écran à 249,99 €

J’ai un code promo de 12 %, puis je dois ajouter une TVA de 20 %.
Les frais de livraison sont de 8,90 €, mais ils sont offerts si le total TTC avant livraison dépasse 350 €.

Quel est le montant final à payer ?
"""

## 10. Premier appel à OpenAI : laisser le modèle choisir un tool

On envoie à OpenAI :

- le prompt utilisateur ;
- les tools convertis depuis MCP.

On inspecte ensuite le message du modèle.

In [ ]:
messages = [
    {
        "role": "user",
        "content": SHOP_PROMPT
    }
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=openai_tools
)

assistant_message = response.choices[0].message

pprint(assistant_message.model_dump())

### Observation attendue

Le modèle devrait produire un `tool_call`, par exemple vers :

```text
calculate_cart_total
```

avec des arguments comme :

```python
{
    "item_prices": [79.9, 39.5, 249.99],
    "discount_percent": 12,
    "tax_percent": 20,
    "shipping_fee": 8.9,
    "free_shipping_threshold": 350
}
```

Important :

> OpenAI n'a pas encore exécuté le tool.  
> Il a seulement demandé un appel de tool.

## 11. Exécuter le tool demandé par OpenAI via MCP

On prend le `tool_call` produit par OpenAI, puis on appelle le serveur MCP avec le même nom de tool et les mêmes arguments.

In [ ]:
if not assistant_message.tool_calls:
    raise RuntimeError("Le modèle n'a pas demandé de tool_call. Relancez la cellule ou forcez tool_choice si besoin.")

tool_call = assistant_message.tool_calls[0]

tool_name = tool_call.function.name
tool_arguments = json.loads(tool_call.function.arguments)

print("Tool demandé par OpenAI :", tool_name)
print("Arguments extraits par OpenAI :")
pprint(tool_arguments)

mcp_output = await call_mcp_tool(tool_name, tool_arguments)

print("\nRésultat renvoyé par le serveur MCP :")
pprint(mcp_output)

### Questions

1. Qui a choisi le tool ?
2. Qui a extrait les arguments ?
3. Qui a exécuté le code réel ?
4. Qui a produit le résultat numérique ?

## 12. Renvoyer le résultat MCP à OpenAI

Maintenant que le tool a été exécuté, on renvoie le résultat à OpenAI.

Le modèle peut alors produire une réponse finale en langage naturel.

In [ ]:
def assistant_message_to_dict(message):
    data = {
        "role": "assistant",
        "content": message.content
    }

    if message.tool_calls:
        data["tool_calls"] = [tool_call.model_dump() for tool_call in message.tool_calls]

    return data


messages.append(assistant_message_to_dict(assistant_message))

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(mcp_output, ensure_ascii=False)
})

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=openai_tools
)

print(final_response.choices[0].message.content)

## 13. Fonction utilitaire : boucle MCP + OpenAI

On regroupe maintenant les étapes précédentes dans une fonction.

Cette fonction :

1. découvre les tools MCP ;
2. les convertit en tools OpenAI ;
3. envoie le prompt à OpenAI ;
4. exécute les tool calls via MCP ;
5. renvoie les résultats à OpenAI ;
6. s'arrête quand le modèle produit une réponse finale.

C'est une mini-boucle agentique.

In [ ]:
async def ask_openai_using_mcp(prompt: str, model: str = MODEL):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = [mcp_tool_to_openai_tool(tool) for tool in tools_result.tools]

            messages = [
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            while True:
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=openai_tools
                )

                assistant_message = response.choices[0].message
                messages.append(assistant_message_to_dict(assistant_message))

                if not assistant_message.tool_calls:
                    return assistant_message.content

                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name
                    arguments = json.loads(tool_call.function.arguments)

                    print(f"Tool demandé : {tool_name}")
                    print("Arguments :")
                    pprint(arguments)

                    mcp_result = await session.call_tool(tool_name, arguments=arguments)
                    mcp_output = mcp_result_to_python(mcp_result)

                    print("Résultat MCP :")
                    pprint(mcp_output)
                    print("-" * 80)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(mcp_output, ensure_ascii=False)
                    })

## 14. Tester la boucle complète

On appelle maintenant notre fonction utilitaire.

In [ ]:
answer = await ask_openai_using_mcp(SHOP_PROMPT)

print("\nRéponse finale :")
print(answer)

## 15. Tester la gestion d'erreur via OpenAI + MCP

Le modèle doit choisir `divide`, puis le serveur MCP doit retourner une erreur structurée.

OpenAI doit ensuite expliquer l'erreur à l'utilisateur.

In [ ]:
ERROR_PROMPT = "Calcule 10 divisé par 0."

answer = await ask_openai_using_mcp(ERROR_PROMPT)

print("\nRéponse finale :")
print(answer)

## 16. Exercice élèves : modifier le serveur MCP

Ajoutez un nouveau tool au fichier `mcp_tp_server.py`.

Exemple : `calculate_rental_price`.

Règles métier :

- une voiture est louée entre une date de début et une date de fin ;
- tout jour commencé est facturé ;
- tarif de base par jour ;
- assurance par jour ;
- supplément jeune conducteur si l'âge du conducteur est inférieur à 25 ans ;
- frais de dossier fixes ;
- remise de 10 % si la location dure au moins 5 jours commencés.

Signature suggérée :

```python
@mcp.tool()
def calculate_rental_price(
    start_datetime: str,
    end_datetime: str,
    driver_age: int,
    daily_price: float,
    insurance_per_day: float,
    young_driver_fee_per_day: float,
    booking_fee: float,
) -> dict:
    ...
```

Puis relancez les cellules de découverte MCP.

Questions :

1. Le nouveau tool apparaît-il automatiquement dans `list_tools()` ?
2. Le schema d'entrée est-il généré automatiquement ?
3. OpenAI peut-il choisir ce nouveau tool sans qu'on écrive manuellement son schema OpenAI ?

## 17. Conclusion pédagogique

Dans le function calling simple, on écrit nous-mêmes la liste des tools dans le code client.

Avec MCP, le serveur expose ses tools, et le client peut les découvrir.

On a donc deux niveaux :

```text
Function calling :
Le modèle demande l'appel d'une fonction.

MCP :
Un serveur expose des fonctions de manière standardisée à des clients compatibles.
```

Dans ce TP, OpenAI ne parle pas directement au serveur MCP local.  
Le notebook joue le rôle de pont :

```text
OpenAI tool_call
    ↓
Notebook / client MCP
    ↓
Serveur MCP local
    ↓
Résultat du tool
    ↓
Notebook
    ↓
OpenAI réponse finale
```

La phrase à retenir :

> MCP standardise l'accès aux outils ; le function calling permet au modèle de demander leur utilisation.